# Import setting

In [ ]:
import os

print(os.getcwd())
if not os.getcwd().endswith("app"):
    os.chdir("../app")
    print(os.getcwd())

%load_ext autoreload
%autoreload 2

from src.config import Configuration

CONFIG = Configuration(
    batch_size=8, # que sino no caben los 3
)

/home/turbotowerlnx/Documents/Master/VPC/VPC-Labs/notebooks
/home/turbotowerlnx/Documents/Master/VPC/VPC-Labs/app
________________________________________________________________________________________________________________________________
                                                         CONFIGURATION                                                          

Experiment description: Base experiment description
Experiment name: base_name
seed: 42
batch_size: 4
epochs: 100
dropout_rate: 0.5
label_smoothing: 0.1
learning_rate: 0.01
weight_decay: 0.0001
eta_min: 1e-06
momentum: 0.9
lr_reduce_factor: 0.5
lr_patience: 3
patience: 10



# Load data

In [2]:
from src.data import load_gender_data

train_loader, test_loader = load_gender_data(CONFIG)

# Load models

In [3]:
import torch
from src.models import GenderModule


top_models = [
    "gender-seed=50-val_acc=0.9456.ckpt",
    "gender-seed=66-val_acc=0.9641.ckpt",
    "gender-seed=72-val_acc=0.9539.ckpt"
]

loaded_models = []
for ckpt in top_models:
    lightning_model = GenderModule.load_from_checkpoint(
        os.path.join(CONFIG.MODELS_PATH, 'bests', ckpt),
        CONFIG=CONFIG,
        map_location=CONFIG.device,
    )
    lightning_model.eval()
    lightning_model.to(CONFIG.device)
    loaded_models.append(lightning_model)

print(f"Loaded {len(loaded_models)} models")

Loaded 3 models


In [ ]:
@torch.no_grad()
def predict_probs_tta(lightning_model, images):
    # Batch-wise TTA: average probs from original and horizontally flipped images.
    logits = lightning_model(images)
    probs = torch.softmax(logits, dim=1)

    images_flipped = torch.flip(images, dims=[3])  # flip width axis (N, C, H, W)
    logits_flipped = lightning_model(images_flipped)
    probs_flipped = torch.softmax(logits_flipped, dim=1)

    return (probs + probs_flipped) / 2.0


@torch.no_grad()
def collect_binary_outputs(lightning_model, data_loader, device):
    labels_all = []
    pos_probs_all = []

    for images, labels in data_loader:
        images = images.to(device)
        probs = predict_probs_tta(lightning_model, images)
        pos_probs = probs[:, 1]  # probability of class 1

        labels_all.append(labels.cpu().long())
        pos_probs_all.append(pos_probs.cpu())

    return torch.cat(labels_all), torch.cat(pos_probs_all)


def find_best_threshold(labels, pos_probs, num_thresholds=1001):
    thresholds = torch.linspace(0.0, 1.0, num_thresholds)
    preds = (pos_probs.unsqueeze(0) >= thresholds.unsqueeze(1)).long()
    accs = (preds == labels.unsqueeze(0)).float().mean(dim=1)

    best_idx = torch.argmax(accs)
    return thresholds[best_idx].item(), accs[best_idx].item()


@torch.no_grad()
def evaluate_ensemble(models, data_loader, device):
    correct = 0
    total = 0
    for images, labels in data_loader:
        images = images.to(device)
        labels = labels.to(device)

        probs_sum = None
        for m in models:
            probs = predict_probs_tta(m, images)
            probs_sum = probs if probs_sum is None else probs_sum + probs

        probs_avg = probs_sum / len(models)
        ensemble_preds = torch.argmax(probs_avg, dim=1)
        correct += (ensemble_preds == labels).sum().item()
        total += labels.numel()

    return correct / total if total > 0 else 0.0


# Individual model threshold search (with TTA)
individual_results = []
for i, m in enumerate(loaded_models, start=1):
    labels_all, pos_probs_all = collect_binary_outputs(m, test_loader, CONFIG.device)
    best_threshold, best_acc = find_best_threshold(labels_all, pos_probs_all)
    individual_results.append({
        "model": i,
        "best_threshold": best_threshold,
        "best_acc": best_acc,
    })
    print(
        f"model_{i} best threshold (TTA): {best_threshold:.3f} | "
        f"accuracy: {best_acc:.4f}"
    )


# Soft-voting ensemble accuracy (with TTA, argmax)
ensemble_acc = evaluate_ensemble(loaded_models, test_loader, CONFIG.device)
print(f"ensemble accuracy (TTA): {ensemble_acc:.4f}")

model_1 best threshold (TTA): 0.464 | accuracy: 0.9468
model_2 best threshold (TTA): 0.511 | accuracy: 0.9668
model_3 best threshold (TTA): 0.467 | accuracy: 0.9577
